# 扩散语言模型与并行生成

> 在之前的章节里，我们已经看到自回归模型生成文本时必须按顺序逐个产出 Token。举例来说，输出一段长度为 200 token 的回复，模型就要做 200 次前向传播，每次只预测一个 token。这种串行特性由 next token prediction 任务本身决定，即便 GPU 算力充裕也无法绕过。
>
> 图像生成则不存在这个限制。以 Stable Diffusion 为例，它从噪声中一次性还原整张图像，通常几十步迭代便可完成，不必从某个角落的像素开始顺序绘制。那么，文本是否也能“整段同时浮现”？问题很明确：图像像素是连续数值，可以逐步添加或去除高斯噪声；而 token 属于离散类别，给某个词“加一点噪声”在词表里找不到对应项，操作本身没有定义。
>
> 本附录将讨论研究者如何跨越这一障碍。我们梳理扩散语言模型的五组核心内容：
>
> 1. **自回归的串行瓶颈**：理解为何必须顺序生成，以及朴素非自回归生成（NAR）为何失败。
> 2. **图像 diffusion 的最小心智模型**：借助连续像素的逐步加噪与去噪，为文本方法铺垫直觉。
> 3. **把噪声换成 [MASK]**：用特殊符号模拟噪声，将 diffusion 思想引入语言，训练目标近似 BERT 的 MLM。
> 4. **手算与玩具实现**：通过排序任务的三步生成，弄清置信度优先解码的机制。
> 5. **实验与工业坐标**：把步数作为质量调节旋钮，并对比 LLaDA、Mercury 等实际模型。
>
> 学完本附录，你将掌握 diffusion LM 如何利用 [MASK] 完成并行生成，并客观认识它在质量与速度上相对于自回归模型的真实差异。

## 0. 自回归的串行瓶颈

前面「解码策略」与「投机解码」两章处理的其实是同一现象：自回归模型要产生长度 $L$ 的序列，就必须执行 $L$ 次前向传播，且步骤不能并行，第 $i$ 个 token 的概率分布依赖于前 $i-1$ 个 token。为什么会这样？因为模型在推算当前词之前，必须等前面的词已经确定。

投机解码采用“一次前向验证多个候选”的思路，平均能减少一些步数，但并没有打破串行的根本结构。那么，怎样才能真正摆脱串行限制？答案在于更换生成范式。

**非自回归生成（Non-Autoregressive Generation, NAR）**指的是不遵循从左至右顺序、通过一次或少量前向传播同时得到全部 token 的生成方法。换句话说，它将“逐字续写”转变为“整段先写再修订”。

这个方法设想很好，可朴素的 NAR（单次前向直接输出所有 token）在自然语言任务上质量糟糕。为何会差？具体原因我们需要用手算例子来看清。

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("使用的设备：", device)

In [ ]:
# 玩具例子：某地餐厅评价的语料里只有两种固定搭配
# 「好吃不贵」和「难吃且贵」——两个空总是一起出现，彼此相关

p_slot1 = {"好": 0.5, "难": 0.5}
p_slot2_given = {
    "好": {"不": 0.9, "且": 0.1},   # 前面是「好吃」，后面大概率接「不贵」
    "难": {"且": 0.9, "不": 0.1},   # 前面是「难吃」，后面大概率接「且贵」
}

# 一步生成：两个空必须同时填，第二个空只能用边缘分布
p_slot2_marginal = {"不": 0.5, "且": 0.5}
one_shot_ok = (p_slot1["好"] * p_slot2_marginal["不"]
               + p_slot1["难"] * p_slot2_marginal["且"])

# 从左到右生成：填第二个空时已经看到第一个空
ar_ok = (p_slot1["好"] * p_slot2_given["好"]["不"]
         + p_slot1["难"] * p_slot2_given["难"]["且"])

print(f"一步生成（两个空独立填）：合法搭配概率 = {one_shot_ok:.0%}")
print(f"从左到右生成（先填第一个空）：合法搭配概率 = {ar_ok:.0%}")
print()
print("关键观察：每个空单独看都有 90% 的把握，独立填却只有 50% 合法。")
print("          一步生成的问题不在「单个词选错」，而在「组合不搭配」。")

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.bar(["one-shot\n(independent slots)", "left-to-right\n(AR)"],
       [one_shot_ok, ar_ok], color=["steelblue", "tomato"], width=0.5)
ax.set_ylim(0, 1)
ax.set_ylabel("probability of a valid phrase")
ax.set_title("Why one-shot generation fails: slots must coordinate")
for i, v in enumerate([one_shot_ok, ar_ok]):
    ax.text(i, v + 0.03, f"{v:.0%}", ha="center")
plt.show()

在那个简单的玩具例子中，仅仅两个空缺，正确率就已经从 90% 下滑到 50%。为何如此糟糕？真实句子里存在几十甚至上百个相互依赖的位置，一步生成带来的“组合不搭配”问题会被成倍放大。

我们将四条技术路线并列比较：

| 生成方式 | 前向传播次数 | 位置之间怎么配合 |
|:---|:---|:---|
| 自回归 | $L$ 次，串行 | 天然配合：右边总是看得到左边 |
| 投机解码 | 约 $L/\text{接受长度}$，仍是串行框架 | 配合方式同自回归 |
| 一步 NAR | 1 次 | 不配合：各填各的，容易组合失调 |
| Masked Diffusion | $k$ 次，$k$ 可调 | 部分配合：每一步都重新看到全局 |

Masked Diffusion 是 NAR 的改进版本。它承认单次生成难以全对，因此改为多轮进行，每轮只确定最有把握的部分，其余留到后续轮次。也就是说，它正是本附录讨论的核心对象。

## 1. 图像 diffusion 的最小心智模型

**Diffusion Model（扩散模型）**是一类定义了“逐步加噪”与“逐步去噪”两个相反过程、并通过学习去噪过程来完成生成的模型。为何这样设计？原因在于它不直接训练“如何生成”，而是训练“如何将打乱的内容复原”。只要学会复原，自然就能生成：从纯噪声起步逐步复原，最终便得到新图像。

这两个过程具体如下：

- **Forward（前向加噪）**：对真实图片不断叠加高斯噪声。噪声强度由时间步 $t$ 控制，$t=0$ 是原图，$t$ 越大越乱，$t=1$ 时几乎是纯噪声
- **Reverse（反向去噪）**：训练一个网络，输入带噪图片和 $t$，预测「干净的样子」。采样时从纯噪声出发，反复调用网络，一小步一小步走回干净图片

In [ ]:
# 用一张 24x24 的合成图像演示 forward 过程：图案逐步被噪声淹没
img = np.zeros((24, 24))
img[8:16, 4:20] = 1.0   # 横条
img[4:20, 10:14] = 1.0  # 竖条，合成一个十字

rng = np.random.default_rng(42)
fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
for ax, t in zip(axes, [0.0, 0.2, 0.5, 1.0, 2.0]):
    noisy = np.clip(img + rng.normal(0, t, img.shape), 0, 1)
    ax.imshow(noisy, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"noise level t = {t:.1f}")
    ax.axis("off")
plt.suptitle("Forward process: an image is gradually destroyed by noise", y=1.04)
plt.show()

print("关键观察：reverse 过程就是从最右边走回最左边。")
print("          每一步的更新是全图同时进行的，去噪网络看得到当前整张图。")

这里的并行能力并非工程优化手段，而是该生成方式的内在属性。为什么？因为每一次更新、每一个像素的推算都利用了全局信息。

这一特性对文本处理十分关键。回想 Self-Attention 章节：如果移除 causal mask，每个位置便能观测全部位置。也就是说，双向注意力构成了文本领域 diffusion 方法的基础。

## 2. 把噪声换成 [MASK]

图像可以添加高斯噪声，这是由于像素值是连续数字，加噪后仍是合法数字。那么 token 为何不能如法炮制？词表是一个离散集合，若给 `"猫"` 叠加 0.1 倍的噪声，得到的结果不在词表中，该操作缺乏定义。

**Masked Diffusion（掩码扩散）**将“加噪”定义为按一定概率把 token 替换为特殊符号 [MASK]，将“去噪”定义为预测被 [MASK] 位置原本的 token 的扩散模型。通俗来讲，就是把句子随机挖空再由模型填补；挖空比例越高意味着噪声越强；全部挖空就等同于纯噪声。

对照图像 diffusion 的两个过程：

- **Forward**：每个位置独立地以概率 $t$ 被替换成 [MASK]。$t=0$ 是原句，$t=1$ 是一整行 [MASK]
- **Reverse**：训练一个双向模型，输入带 [MASK] 的句子，在每个 [MASK] 位置输出词表上的概率分布

In [ ]:
# 演示 forward 过程作用在文本上的样子：█ 代表 [MASK]
demo_text = "12+34=046"
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 4, figsize=(11, 2.0))
for ax, t in zip(axes, [0.25, 0.5, 0.75, 1.0]):
    shown = [c if rng.random() > t else "█" for c in demo_text]
    for col, ch in enumerate(shown):
        if ch == "█":
            face, edge = "#cccccc", "none"
        else:
            face, edge = "#e8f0fe", "#7a9cc6"
        ax.text(col, 0, ch, ha="center", va="center", fontsize=13,
                bbox=dict(boxstyle="square,pad=0.4", facecolor=face, edgecolor=edge))
    ax.set_xlim(-0.8, len(demo_text) - 0.2)
    ax.set_ylim(-0.6, 0.6)
    ax.axis("off")
    ax.set_title(f"mask rate t = {t:.2f}", fontsize=10)
plt.suptitle("Forward process on text: masking plays the role of noise", y=1.12)
plt.show()

print("关键观察：t 从小到大，句子逐渐被 [MASK] 淹没；t=1 时信息全部丢失。")
print("          生成就是反着走：从全 [MASK] 出发，一步步把信息恢复出来。")

读者或许已经注意到，上述训练目标与 BERT 章节的 MLM（Masked Language Modeling）几乎一致。事实确实如此，差别只在于使用方式：

| | BERT 的 MLM | Masked Diffusion |
|:---|:---|:---|
| 训练时 mask 比例 | 固定 15% | 从 0 到 100% 均匀采样 |
| 预测几次 | 一次 | 采样时迭代多轮 |
| 注意力 | 双向 | 双向 |

BERT 仅挖空一次并填充一次，属于理解类任务；diffusion 则将“挖空与填补”构建为一条可行走的路径，从全空抵达全满，这才构成生成。

**反向过程（生成）**始于全部为 [MASK] 的答案区，每一轮执行三个步骤。首先把当前序列输入模型，得到每个 [MASK] 位置的概率分布。接着，将每个位置的最大概率称为**置信度**，代表模型对该位置的把握程度。然后，揭晓置信度最高的 $k$ 个位置，未选中的继续保留为 [MASK]，进入下一轮。

上述做法即为**置信度优先解码**：优先完成有把握的判断，把最不确定的位置留到后续信息更充分的轮次。工业界将“每步选择哪些位置揭晓”统称为 remasking 策略，置信度优先只是其中一种，第 6 节会列出完整家族。

若借考试来说明：先写完有把握的题目，难题暂空；首轮完成后，已写答案提供新线索，再回头处理空白题。

## 3. 手算验证：一次 3 步生成

我们先明确规则：**每步揭晓几个**。假设剩余 $m$ 个 [MASK]、还剩 $s$ 步（含当前步），则每步揭晓 $k=\lceil m/s \rceil$ 个。为何取向上取整？是为了保证恰好在预定步数内完成。

任务是对 `9 2 5 1` 排序。完整序列为 `9 2 5 1=1 2 5 9`，`=` 之后是 7 个字符的答案区（4 个数字 + 3 个空格）。设定 $T=3$ 步生成，输入区（`=` 之前）始终可见。

**第 1 步**：7 个位置全是 [MASK]，$k=\lceil 7/3\rceil=3$。模型具备双向能力，能观测输入区的 `9 2 5 1`：

| 位置 | 内容 | 模型的概率（示意） | 置信度 |
|:--|:--|:--|:--|
| 0 | [MASK] | `1`: 0.60，`2`: 0.25，其他 0.15 | 0.60 |
| 1 | [MASK] | `空格`: 0.98，`5`: 0.01，… | **0.98** |
| 2 | [MASK] | `2`: 0.45，`5`: 0.40，… | 0.45 |
| 3 | [MASK] | `空格`: 0.98，… | **0.98** |
| 4 | [MASK] | `5`: 0.45，`2`: 0.40，… | 0.45 |
| 5 | [MASK] | `空格`: 0.98，… | **0.98** |
| 6 | [MASK] | `9`: 0.75，`5`: 0.15，… | 0.75 |

空格的排列模式固定，置信度接近 1，因此最优先揭晓。数字中最小值 `1` 与最大值 `9` 较易识别，中间的 `2` `5` 最模糊。于是揭晓位置 1、3、5，答案区变为 `[M]空[M]空[M]空[M]`。

**第 2 步**：剩余 4 个数字位，$k=\lceil 4/2\rceil=2$。需注意概率已变化：空格确定后模式更清晰：

| 位置 | 概率（示意） | 置信度 | |
|:--|:--|:--|:--|
| 0 | `1`: 0.90，… | **0.90** | 揭晓 |
| 2 | `2`: 0.50，`5`: 0.42，… | 0.50 | 保留 |
| 4 | `5`: 0.50，`2`: 0.42，… | 0.50 | 保留 |
| 6 | `9`: 0.88，… | **0.88** | 揭晓 |

**第 3 步**：剩余位置 2、4，$k=\lceil 2/1\rceil=2$。此时两端 `1` 和 `9` 已揭晓，输入剩余数字仅有 `2` 和 `5`，顺序被锁定：

| 位置 | 概率（示意） | 置信度 | |
|:--|:--|:--|:--|
| 2 | `2`: 0.95，… | **0.95** | 揭晓 |
| 4 | `5`: 0.97，… | **0.97** | 揭晓 |

手算中最关键的发现是：**位置 2 的置信度从第 1 步的 0.45 涨到了第 3 步的 0.95**。该位置本身未变，变化的是它的上下文：每轮揭晓的 token 都成为新的条件。也就是说，迭代去噪的价值就在于把最困难的决定推迟到信息最丰富的时刻。

反过来看，也能理解一步生成的困境：模型不得不在信息最少（全 [MASK]）时做出所有决定。这正是下一节实验中一步质量崩塌的缘由。

## 4. 代码实现：Mini Masked Diffusion LM

我们选定**排序**作为任务：输入 8 个 0-9 的随机数字，输出其升序排列。完整序列类似 `8 6 5 2 3 0 0 0=0 0 0 2 3 5 6 8`。为何选它？理由有三点：其一，答案区长达 15 个 token，自回归需 15 次前向，并行生成的收益明显；其二，正确性可自动验证，整条序列与标准答案完全相同才算 exact match；其三，排序依赖全局信息，每个输出位置都需整个输入，双向注意力确实在发挥作用。

模型直接复用「从零实现 GPT」一节的结构，只有一处差别：**去掉 causal mask**。另外我们不给模型注入时间步 $t$ 的 embedding，玩具任务上模型自己能应付不同的 mask 比例；真实模型的做法见第 6 节。

In [ ]:
def make_sample(rng, n_nums=8):
    """随机生成 n_nums 个 0-9 的数字，返回「乱序=升序」格式的字符串"""
    nums = list(rng.integers(0, 10, size=n_nums))
    left = " ".join(str(d) for d in nums)
    right = " ".join(str(d) for d in sorted(nums))
    return f"{left}={right}"

CHARS = sorted(set("0123456789 ="))
char2id = {c: i for i, c in enumerate(CHARS)}
MASK_ID = len(CHARS)           # [MASK] 排在词表最后
VOCAB = len(CHARS) + 1

def encode(s):
    return [char2id[c] for c in s]

def decode(ids):
    """把 id 序列还原成字符串；[MASK] 显示成 █"""
    table = {i: c for c, i in char2id.items()}
    return "".join(table.get(i, "█") for i in ids)

rng = np.random.default_rng(1)
data = torch.tensor([encode(make_sample(rng)) for _ in range(20000)])

SEQ_LEN = data.shape[1]         # 8 个数字 + 7 个空格 + = + 15 个答案字符 = 31
ANS_START = 16                  # '=' 之后第一个位置，从这里开始是答案区
ANS_LEN = SEQ_LEN - ANS_START   # 15

print("样例：", decode(data[0].tolist()))
print(f"序列长度 {SEQ_LEN}，答案区 {ANS_LEN} 个位置，词表大小 {VOCAB}（含 [MASK]）")
print("关键观察：答案区 15 个位置，自回归要 15 次前向；下面看 diffusion 需要几次")

下面看模型定义。对比「从零实现 GPT」章节的 Mini-GPT，唯一改动是 `bidirectional` 参数：当设为 `True` 时不再传入 causal mask，各位置彼此可见。也就是说，模型从此能够同时观测前文与后文。

In [ ]:
class Block(nn.Module):
    """标准 Transformer Block：Pre-LN + Self-Attention + MLP"""

    def __init__(self, d, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(
            nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=attn_mask)
        x = x + a
        return x + self.mlp(self.ln2(x))


class TinyTransformer(nn.Module):
    """小 Transformer。bidirectional=True 时无 causal mask（diffusion 用），
    False 时加 causal mask（自回归基线用）"""

    def __init__(self, d=128, n_layer=3, n_head=4, bidirectional=True):
        super().__init__()
        self.tok = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(40, d)
        self.blocks = nn.ModuleList([Block(d, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, VOCAB)
        self.bidirectional = bidirectional

    def forward(self, idx):
        T = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        if self.bidirectional:
            mask = None
        else:
            # 上三角为 True = 禁止看向未来：这就是 causal mask
            mask = torch.triu(
                torch.ones(T, T, dtype=torch.bool, device=idx.device), diagonal=1
            )
        for b in self.blocks:
            x = b(x, mask)
        return self.head(self.ln_f(x))

In [ ]:
torch.manual_seed(42)
diff_model = TinyTransformer(bidirectional=True).to(device)
print("参数量：", sum(p.numel() for p in diff_model.parameters()))

训练循环的本质是“挖空与补空”，与 BERT 的区别仅在于 mask 比例随机：

1. 每条序列采样一个噪声水平 $t \sim U(0,1)$
2. 每个位置独立地以概率 $t$ 被替换成 [MASK]
3. 交叉熵只在被 mask 的位置计算，模型学的就是「看上下文补空」

在 Apple Silicon GPU 上训练约 6 分钟，纯 CPU 会明显更久。这段时间用来做什么？正好可以回顾第 3 节的手算过程。

In [ ]:
def train_diffusion(model, steps=3000, bs=128, lr=1e-3):
    """训练 masked diffusion 模型。返回 (step, loss) 列表用于画曲线"""
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    for step in range(steps):
        batch = data[torch.randint(0, len(data), (bs,))].to(device)
        t = torch.rand(bs, 1, device=device)               # 每条序列一个噪声水平
        mask = torch.rand(bs, SEQ_LEN, device=device) < t  # True 的位置将被挖空
        if not mask.any():
            continue                                        # 小概率一个都没挖到
        x = batch.clone()
        x[mask] = MASK_ID
        logits = model(x)
        loss = F.cross_entropy(logits[mask], batch[mask])   # 只在挖空处算损失
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (step + 1) % 500 == 0:
            print(f"  step {step + 1:5d}  loss {loss.item():.4f}")
            hist.append((step + 1, loss.item()))
    return hist

In [ ]:
t0 = time.time()
diff_hist = train_diffusion(diff_model, steps=3000)
print(f"训练完成，用时 {time.time() - t0:.0f} 秒")

随后实现解码循环。对照第 3 节的手算：预测所有 [MASK] 位置、按置信度揭晓 top-$k$、其余留到下一轮。也就是说，将之前纸面推演的三步转化为代码。

In [ ]:
@torch.no_grad()
def diffuse_generate(model, prompt_ids, n_steps, record=False):
    """从「答案区全 [MASK]」开始做 n_steps 步去噪。
    record=True 时额外返回每步的序列快照，用于可视化"""
    model.eval()
    seq = list(prompt_ids) + [MASK_ID] * ANS_LEN
    masked = list(range(ANS_START, SEQ_LEN))
    snaps = [list(seq)]
    for step_i in range(n_steps):
        ids = torch.tensor([seq], device=device)
        logits = model(ids)[0]
        pos = torch.tensor(masked, device=device)
        probs = F.softmax(logits[pos], dim=-1)
        conf, pick = probs.max(dim=-1)                   # 最大概率 = 置信度
        k = math.ceil(len(masked) / (n_steps - step_i))  # 保证 n_steps 步内走完
        top = conf.argsort(descending=True)[:k]          # 置信度最高的 k 个
        for j in top.tolist():
            seq[masked[j]] = pick[j].item()
        chosen = set(top.tolist())
        masked = [masked[j] for j in range(len(masked)) if j not in chosen]
        snaps.append(list(seq))
    assert not masked, "解码结束后不应剩任何 [MASK]"
    return (seq, snaps) if record else seq

In [ ]:
# 固定一批测试样本，后面所有评测都用它们
test_rng = np.random.default_rng(99)
tests = [make_sample(test_rng) for _ in range(200)]

sample = tests[0]
prompt = encode(sample[:ANS_START])
print("输入（可见部分）：", sample[:ANS_START])
print("标准答案：", sample[ANS_START:])
seq, snaps = diffuse_generate(diff_model, prompt, n_steps=5, record=True)
print("生成结果：", decode(seq)[ANS_START:])
print("整条全对：", decode(seq) == sample)
print()
for i, snap in enumerate(snaps):
    label = "初始" if i == 0 else f"第 {i} 步"
    print(f"  {label}: {decode(snap)[ANS_START:]}")

## 5. 实验观察

我们先把前述 5 步生成的过程可视化：每一行代表一个解码时刻，每一列对应答案区的一个位置，█ 表示尚未揭晓的 [MASK]。为何要画出？因为这样能直观展现逐步揭晓的节奏。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
grid = np.zeros((len(snaps), ANS_LEN))
for r, snap in enumerate(snaps):
    for c, ch in enumerate(decode(snap)[ANS_START:]):
        grid[r, c] = 0.0 if ch == "█" else 1.0
ax.imshow(grid, cmap="Blues", aspect="auto", vmin=0, vmax=1)
for r, snap in enumerate(snaps):
    for c, ch in enumerate(decode(snap)[ANS_START:]):
        ax.text(c, r, ch, ha="center", va="center", fontsize=10,
                color="gray" if ch == "█" else "black")
ax.set_yticks(range(len(snaps)))
ax.set_yticklabels(["init"] + [f"step {i + 1}" for i in range(len(snaps) - 1)])
ax.set_xlabel("answer position")
ax.set_title("Masked diffusion decoding: positions get revealed over steps")
plt.show()
print("关键观察：空格位置最先被填上（最容易），数字按置信度逐步揭晓。")
print("          每一步都在「整段一起写」，而不是从左到右逐字符写。")

再训练一个自回归基线用于对比。使用同一个 TinyTransformer，设置 `bidirectional=False`，训练目标为标准的 next token prediction；生成时从左向右，每个 token 需一次前向。

In [ ]:
def train_ar(model, steps=3000, bs=128, lr=1e-3):
    """训练自回归基线：全部位置都算 next token 的交叉熵"""
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    for step in range(steps):
        batch = data[torch.randint(0, len(data), (bs,))].to(device)
        logits = model(batch[:, :-1])
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB), batch[:, 1:].reshape(-1)
        )
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (step + 1) % 500 == 0:
            print(f"  step {step + 1:5d}  loss {loss.item():.4f}")
            hist.append((step + 1, loss.item()))
    return hist


@torch.no_grad()
def ar_generate(model, prompt_ids):
    """自回归生成：每次前向只为拿下一个 token，共 ANS_LEN 次前向"""
    model.eval()
    seq = list(prompt_ids)
    for _ in range(ANS_LEN):
        ids = torch.tensor([seq], device=device)
        seq.append(int(model(ids)[0, -1].argmax()))
    return seq

In [ ]:
torch.manual_seed(42)
ar_model = TinyTransformer(bidirectional=False).to(device)
t0 = time.time()
ar_hist = train_ar(ar_model, steps=3000)
print(f"训练完成，用时 {time.time() - t0:.0f} 秒")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
dx, dy = zip(*diff_hist)
ax_, ay = zip(*ar_hist)
ax.plot(dx, dy, "o-", ms=4, label="masked diffusion (loss on masked positions)")
ax.plot(ax_, ay, "s-", ms=4, label="autoregressive (loss on all positions)")
ax.set_xlabel("training step")
ax.set_ylabel("cross-entropy loss")
ax.set_title("Both objectives converge")
ax.legend()
plt.show()
print("关键观察：两条 loss 各自趋平即收敛。数值不能横向比：")
print("          diffusion 的 loss 只算被挖空的位置，天然包含高噪声的难题")

重头戏是**步数与质量的关系**。对同一批 200 个测试样本，分别用 1、2、3、5、8、15 步生成，统计整条序列全对的比例；自回归基线固定 15 次前向。也就是说，我们想观察增加步数能否挽回质量。

In [ ]:
def acc_diffusion(n_steps, n=200):
    ok = 0
    for s in tests[:n]:
        gen = diffuse_generate(diff_model, encode(s[:ANS_START]), n_steps)
        ok += (decode(gen) == s)
    return ok / n


def acc_ar(n=200):
    ok = 0
    for s in tests[:n]:
        gen = ar_generate(ar_model, encode(s[:ANS_START]))
        ok += (decode(gen) == s)
    return ok / n


step_list = [1, 2, 3, 5, 8, 15]
diff_accs = [acc_diffusion(ns) for ns in step_list]
ar_acc = acc_ar()

for ns, a in zip(step_list, diff_accs):
    print(f"diffusion {ns:2d} 步：整条全对率 {a:.1%}")
print(f"自回归    {ANS_LEN} 步：整条全对率 {ar_acc:.1%}")
print()
print(f"关键观察：{step_list[0]} 步只有 {diff_accs[0]:.0%}，"
      f"{step_list[-1]} 步升到 {diff_accs[-1]:.0%}——步数就是质量旋钮")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

ax1.plot(step_list, diff_accs, "o-", label="masked diffusion")
ax1.axhline(ar_acc, color="tomato", ls="--",
            label=f"autoregressive ({ANS_LEN} fwd passes)")
ax1.set_xlabel("number of diffusion steps")
ax1.set_ylabel("exact match accuracy")
ax1.set_title("Quality vs number of steps")
ax1.set_ylim(0, 1.05)
ax1.legend()

labels = ["AR"] + [f"{ns}-step" for ns in step_list]
fwd = [ANS_LEN] + step_list
ax2.bar(labels, fwd, color=["tomato"] + ["steelblue"] * len(step_list))
ax2.set_ylabel("forward passes per generation")
ax2.set_title("Cost: forward passes")
plt.tight_layout()
plt.show()

实验给出三个读数（百分比会随训练随机性浮动，你复跑时数字会不同，趋势应当一致）：

- **1 步明显落后**：全对比例仅约一半，而自回归接近 100%。为何这么低？该任务为确定性任务，给定输入仅有唯一正确答案，因此并非第 0 节提及的“组合不搭配”，而是模型容量有限时单次前向无法完成全部推理，且缺乏修正机会。
- **步数是质量旋钮**：从 1 步到 15 步，正确率整体攀升（个别档位小幅波动源于 200 个样本的统计噪声）。若想前向次数减至五分之一，可用 3 步，但需接受约一半正确率；若追求质量则增加步数。这种“生成时才选定档位”的自由度是自回归不具备的。
- **全步数仍逊于自回归**：15 步 diffusion 与自回归前向次数相同，质量却仍逊一筹。这与工业界公开结论吻合：diffusion LM 已逼近同规模 AR 模型，但尚未持平。

| | 自回归 | 一步 NAR | Masked Diffusion |
|:---|:---|:---|:---|
| 前向次数 | $L$，串行 | 1 | $k$，可调 |
| 位置配合 | 从左到右条件生成 | 无 | 每步全局重看 |
| 质量上限 | 最高 | 最低 | 接近但低于自回归 |
| 独有自由度 | 无 | 无 | 用步数换速度的连续旋钮 |

## 6. 和工业界的区别

**真实模型长什么样。** 典型代表是 LLaDA（2025，8B）：它从 LLaMA 3 初始化并继续训练为 masked diffusion 模型，在多项通用基准上接近同规模 AR 模型。玩具版本省略了其两处改进：

- 训练目标不是朴素交叉熵，而是带似然权重的 ELBO，不同噪声水平 $t$ 的贡献按理论加权
- 把 $t$ 的 embedding 注入模型，让模型明确知道「现在噪声多大」（我们靠模型自己适应）

**remasking 策略全家福：**

| 策略 | 每步揭晓谁 | 一句话点评 |
|:---|:---|:---|
| random | 随机挑 | 最简单，质量最差 |
| confidence | 置信度最高的 | 本附录的实现，最常用 |
| semi-autoregressive | 从左到右按块揭晓 | 保留 AR 的位置感 |
| low-confidence remasking | 每步把低置信度的已揭晓位置重新变回 [MASK] | 允许反悔，质量更高 |

我们实现的版本“揭晓不反悔”；最后一种策略每轮会把模型没把握的已揭晓位置重新挖空，下轮再决策。为何这关键？能够改正错误是 diffusion 相对自回归的结构性优势：AR 一旦生成错误便只能延续，这与「后训练技术演进」章节讨论的 exposure bias 同源。

**推理系统要重新设计。** KV cache 对 diffusion 失效：双向注意力叠加每步整段重算，使缓存结构不复存在。「现代 LLM 推理系统」章节的 continuous batching、调度思想可复用，但瓶颈与优化点不同。

**现状与定位。** 自 2025 年起出现商业落地：Inception Labs 的 Mercury 主攻代码生成（官方数字超过 1000 tokens/s），Google 发布了 Gemini Diffusion。公开评测中 diffusion LM 质量仍普遍弱于同规模 AR，在代码等输出分布集中（低熵）任务上优势最显著。缩放定律、对齐方法、推理引擎等生态均围绕 AR 构建，迁移需时；不少研究者视 AR 打底、diffusion 加速的混合架构为过渡形态。

它是否值得学？回到开篇：串行瓶颈是 next token prediction 的结构性约束，diffusion 是当前唯一进入工业界的“换范式”方案。也就是说，理解它，你就获得了自回归之外的另一参照系。

## 小结

本章内容已结束，我们梳理核心要点。

- 自回归受限于串行瓶颈：产生 $L$ 个 token 必须 $L$ 次顺序前向；投机解码只在框架内缓解，diffusion 才是换框架
- 一步 NAR 失败主因是“组合不搭配”：各位置单独看有把握，合并却非法
- Masked Diffusion 以 [MASK] 模拟噪声：加噪即按比例挖空，去噪靠双向模型补空；训练目标与 BERT 的 MLM 同宗，mask 比例覆盖 0 到 100%
- 解码循环分三步：预测全部 [MASK] 位置、按置信度揭晓 top-$k$、余下留待下轮；$k=\lceil m/s\rceil$ 确保收尾
- 实验结论：步数充当质量旋钮，单调交换质量；全步数仍稍逊自回归，与工业界观察相符
- 工业坐标：LLaDA、Mercury、Gemini Diffusion；remasking 策略族；KV cache 失效引发推理系统重构

若欲深入，LLaDA 论文附录含完整 ELBO 推导；将本附录 `diffuse_generate` 改为 low-confidence remasking 版本，是验证是否掌握解码循环的最佳练习。

## 作业

> 允许使用 AI 探询思路、拆解步骤、核查方向，但不建议直接让 AI 「做完这道题」。

动手实践才能真懂。以下三题均基于已训练模型与第 4 节代码，无需重新训练。

1. **实现前向过程的 mask 采样**。小提示：每个位置独立判断，一次 `torch.rand` 加一个比较运算符即可
2. **每步揭晓几个位置**。小提示：15 个位置分 4 步，想想用 floor 会在哪一步出问题
3. **random 揭晓对比 confidence 揭晓**。小提示：`torch.randperm` 给出均匀随机排列

In [ ]:
# 作业 1：实现前向过程的 mask 采样
# 每个位置独立地以概率 t 被 mask——这是 masked diffusion 对「加噪」的定义

def sample_mask(seq_len, t, gen):
    """返回布尔张量：True 表示该位置要替换成 [MASK]
    seq_len：序列长度；t：噪声水平（0~1）；gen：随机数生成器（保证可复现）
    """
    rand = torch.rand(seq_len, generator=gen)
    mask = rand ___ t          # ← 填空：一个比较运算符
    return mask.bool()

g = torch.Generator().manual_seed(0)
m = sample_mask(1000, 0.3, g)
assert m.dtype == torch.bool, "mask 应该是布尔张量"
assert 200 <= int(m.sum()) <= 400, \
    f"t=0.3、长度 1000 期望约 300 个 True，你得到 {int(m.sum())}"
assert sample_mask(1000, 1.0, torch.Generator().manual_seed(1)).all(), \
    "t=1.0 时应该全部被 mask"

print("✅ 作业 1 通过：你已经会给序列采样噪声了")
print("   t 就是 mask 比例：t 越大噪声越强，t=1 时整条序列变成 [MASK]")

In [ ]:
# 作业 2：每步揭晓多少个位置？
# k 取小了走不完，取大了浪费步数

def reveal_count(remaining, steps_left):
    """remaining：还剩多少个 [MASK]；steps_left：还剩多少步（含当前步）
    返回这一步要揭晓的位置数
    """
    return math.___(remaining / steps_left)   # ← 填空：ceil 还是 floor？

# 15 个位置、4 步走完：每步 4 4 4 3
assert reveal_count(15, 4) == 4
assert reveal_count(11, 3) == 4
assert reveal_count(3, 1) == 3

print("✅ 作业 2 通过：向上取整保证恰好走完")
print("   如果用 floor：15//4=3，四步只揭晓 12 个，最后 3 个位置没有步数可用了")

In [ ]:
# 作业 3：把 confidence 揭晓换成 random 揭晓，质量会掉多少？

@torch.no_grad()
def diffuse_generate_random(model, prompt_ids, n_steps, seed=7):
    """与 diffuse_generate 逻辑相同，但每步「随机」挑 k 个位置揭晓"""
    gen = torch.Generator().manual_seed(seed)
    model.eval()
    seq = list(prompt_ids) + [MASK_ID] * ANS_LEN
    masked = list(range(ANS_START, SEQ_LEN))
    for step_i in range(n_steps):
        ids = torch.tensor([seq], device=device)
        logits = model(ids)[0]
        pos = torch.tensor(masked, device=device)
        probs = F.softmax(logits[pos], dim=-1)
        _, pick = probs.max(dim=-1)
        k = math.ceil(len(masked) / (n_steps - step_i))
        top = torch.___(len(masked), generator=gen)[:k].tolist()  # ← 填空
        for j in top:
            seq[masked[j]] = pick[j].item()
        chosen = set(top)
        masked = [masked[j] for j in range(len(masked)) if j not in chosen]
    return seq

n_eval, n_steps = 100, 3
ok_r = ok_c = 0
for s in tests[:n_eval]:
    prompt = encode(s[:ANS_START])
    ok_r += (decode(diffuse_generate_random(diff_model, prompt, n_steps)) == s)
    ok_c += (decode(diffuse_generate(diff_model, prompt, n_steps)) == s)
acc_random, acc_conf = ok_r / n_eval, ok_c / n_eval
print(f"random 揭晓      3 步：正确率 {acc_random:.0%}")
print(f"confidence 揭晓  3 步：正确率 {acc_conf:.0%}")
assert acc_random <= acc_conf + 0.15, \
    "随机揭晓通常不会好于置信度优先（留了少量随机波动的余地）"

print("✅ 作业 3 通过：揭晓顺序本身是有信息量的")
print("   confidence 把容易的决定先做掉，把难的位置留到上下文最全的最后一轮")
print("   remasking 策略研究的核心问题：同样数量的前向传播，怎么分配最划算")

## 参考资料

若想追溯根源，可查阅以下文献：

- Ho et al., 2020, *Denoising Diffusion Probabilistic Models*（DDPM，图像 diffusion 的奠基工作）
- Gu et al., 2018, *Non-Autoregressive Neural Machine Translation*（NAR 生成的起点，组合不搭配问题首次被系统研究）
- Nie et al., 2025, *Large Language Diffusion Models*（LLaDA，本附录主要取材的对象）
- Inception Labs *Mercury* 与 Google *Gemini Diffusion* 的官方发布（2025，工业界 diffusion LM）
- Stanford CME295 (Autumn 2025) Lecture 9 与 Stanford CS336 (Spring 2025) Lecture 10 的相关讲义